In [1]:
from torch.utils.data import DataLoader, random_split 
from firealarm_net import FireAlarmCNN, MelDataset
from sklearn.preprocessing import LabelEncoder 
import torch.nn as nn
import numpy as np 
import torch 
import os

Apple


In [2]:
# Set seed to have the same outputs constantly when testing
seed = 42
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
np.random.seed(seed)

In [3]:
# Setting up our features
# Load Data by traversing through the features
features = []
labels = []

print("Loading data...")
for filename in sorted(os.listdir("features")):
    if filename.endswith(".npy"):
        # Simple label extraction based on filename
        if "fire" in filename: 
            label = "fire_alarm"
        elif "siren" in filename: 
            label = "siren"
        elif "appliance" in filename: 
            label = "appliance"
        else: 
            continue
        
        features.append(np.load(os.path.join("features", filename)))
        labels.append(label)

# Convert to Tensors
features = np.array(features)

# Standardize data (make mean 0, std 1) for better learning
features = (features - features.mean()) / (features.std() + 1e-8)

# Add channel dim: (N, 64, 32) -> (N, 1, 64, 32)
features = features[:, None, :, :]

# Encode Labels (fire_alarm -> 1, siren -> 2, appliance -> 3)
encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(labels)

print(f"Data loaded: {len(features)} samples.")

Loading data...
Data loaded: 2222 samples.


In [4]:
# Preparing for our Model
# Dataset & Loader
dataset = MelDataset(features, labels_encoded)
train_size = int(0.8 * len(dataset))
train_ds, val_ds = random_split(dataset, [train_size, len(dataset) - train_size], generator=torch.Generator().manual_seed(seed))

# Create a generator for DataLoader shuffling
g = torch.Generator()
g.manual_seed(seed)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=g)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

# Model setup
model = FireAlarmCNN(num_classes=len(encoder.classes_))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss() # Standard loss, no weights

/opt/anaconda3/envs/Random/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Training the Model
epochs = 30
best_acc = 0.0

print("\nStarting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for x, y in train_loader:
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        
        # Track training stats
        running_loss += loss.item() * x.size(0)
        correct += (output.argmax(1) == y).sum().item()
        total += y.size(0)
    
    train_loss = running_loss / total
    train_acc = correct / total
    
    # Validation step
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for x, y in val_loader:
            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item() * x.size(0)
            val_correct += (out.argmax(1) == y).sum().item()
            val_total += y.size(0)
            
    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    
    # This is the logging you asked about - kept exactly as requested
    print(f"Epoch {epoch+1}/{epochs}  "
          f"TrainLoss={train_loss:.4f}  ValLoss={val_loss:.4f}  "
          f"TrainAcc={train_acc:.4f}  ValAcc={val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'model/model.pt')

print(f"Done! Best Accuracy: {best_acc:.4f}")


Starting training...
Epoch 1/30  TrainLoss=0.8889  ValLoss=0.7713  TrainAcc=0.5914  ValAcc=0.6944
Epoch 2/30  TrainLoss=0.6860  ValLoss=0.6259  TrainAcc=0.7079  ValAcc=0.7258
Epoch 3/30  TrainLoss=0.6214  ValLoss=0.5960  TrainAcc=0.7355  ValAcc=0.7596
Epoch 4/30  TrainLoss=0.5423  ValLoss=0.5036  TrainAcc=0.7794  ValAcc=0.8539
Epoch 5/30  TrainLoss=0.4988  ValLoss=0.5103  TrainAcc=0.7839  ValAcc=0.7551
Epoch 6/30  TrainLoss=0.4694  ValLoss=0.4805  TrainAcc=0.8025  ValAcc=0.7978
Epoch 7/30  TrainLoss=0.4249  ValLoss=0.4368  TrainAcc=0.8368  ValAcc=0.7910
Epoch 8/30  TrainLoss=0.4150  ValLoss=0.3587  TrainAcc=0.8227  ValAcc=0.8719
Epoch 9/30  TrainLoss=0.3570  ValLoss=0.3726  TrainAcc=0.8644  ValAcc=0.8472
Epoch 10/30  TrainLoss=0.3627  ValLoss=0.3061  TrainAcc=0.8548  ValAcc=0.8899
Epoch 11/30  TrainLoss=0.3347  ValLoss=0.3111  TrainAcc=0.8678  ValAcc=0.8966
Epoch 12/30  TrainLoss=0.3113  ValLoss=0.2863  TrainAcc=0.8779  ValAcc=0.8921
Epoch 13/30  TrainLoss=0.2943  ValLoss=0.2885  Trai